# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook provides a template for loading and exploring a dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
print(f"{dataset.metadata.name}: {dataset.metadata.description}")
print(f"Published on: {dataset.metadata.datePublished}")
print(f"Keywords: {dataset.metadata.keywords}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

All entities are referenced by their `@id`, as specified by the Croissant schema.

Let's list all available record sets and their fields.

In [ ]:
# List all record sets and their fields by @id
record_sets = []
if hasattr(dataset.metadata, 'recordSet'):
    for rs in dataset.metadata.recordSet:
        record_sets.append(rs['@id'])
        print(f"Record set @id: {rs['@id']}")
        if 'field' in rs:
            print("  Fields:")
            for fld in rs['field']:
                print(f"    - {fld['@id']} ({fld.get('name')})")
        else:
            print("  No fields listed.")
else:
    print("No record sets found in metadata.")

# For demo, show first record set and its fields
first_record_set_id = record_sets[0] if record_sets else None

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s from the overview.

Here we extract records from all available record sets, referencing each by `@id`.

In [ ]:
# Extract data from each record set
dataframes = {}
for record_set_id in record_sets:
    print(f"Loading records for Record Set @id: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if len(records) == 0:
        print(f"  No records found for {record_set_id}.")
    else:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"  Fields: {df.columns.tolist()}")
        print(df.head(2))

# Show columns for the first record set
if first_record_set_id in dataframes:
    print(f"Columns for {first_record_set_id}: {dataframes[first_record_set_id].columns.tolist()}")
    display(dataframes[first_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data.
This section includes operations like removing outliers, transforming data distributions, or grouping data by key attributes.

We use the field `@id` as references (see above for names).

In [ ]:
# EDA on the main record set
# Choose a numeric field by @id. Replace below with actual @id from your schema.
main_record_set_id = first_record_set_id
df = dataframes[main_record_set_id]

# Example numeric field IDs (Replace with actual @id from your dataset after overview)
# Common survey field might be: 'GAD7_total_score', 'PHQ9_total_score', 'PSQ_total_score' etc, referenced by their @id
# For demonstration, we select the first numeric column we can find by inspecting the DataFrame types
numeric_field_id = None
for col in df.columns:
    if pd.api.types.is_numeric_dtype(df[col]):
        numeric_field_id = col
        break
if numeric_field_id is None:
    print("No numeric field found in first record set.")
else:
    threshold = df[numeric_field_id].mean()
    filtered_df = df[df[numeric_field_id] > threshold]
    print(f"Filtered records with {numeric_field_id} > {threshold:.2f}:")
    print(filtered_df.head())

    filtered_df[f"{numeric_field_id}_normalized"] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()
    print(f"Normalized {numeric_field_id} for filtered records:")
    print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"].head())

    # Choose a group field (categorical) by @id, e.g., demographic@id
    group_field_id = None
    for col in df.columns:
        if pd.api.types.is_object_dtype(df[col]) and df[col].nunique() < 20:
            group_field_id = col
            break

    if group_field_id:
        grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
        print(f"Grouped data by {group_field_id} (Mean of {numeric_field_id}):")
        print(grouped_df.head())
    else:
        print("No suitable group field found.")

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset (referenced by their `@id`).

In [ ]:
# Simple visualization for numeric field distribution and group comparison
if numeric_field_id is not None:
    plt.figure(figsize=(8, 4))
    sns.histplot(df[numeric_field_id].dropna(), bins=20, kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Frequency")
    plt.show()

    if group_field_id:
        plt.figure(figsize=(10, 4))
        sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id)
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xticks(rotation=45)
        plt.show()

## 6. Conclusion
Summarize key findings and observations from the dataset exploration.

- The FAIR² Kilifi Mental Health Survey dataset was successfully loaded and explored using Croissant and `mlcroissant`.
- Record sets and fields were referenced by their `@id` according to FAIR practice.
- Numeric mental health scores were reviewed, filtered, normalized, and grouped by demographics.
- Basic visualizations provide insights into score distributions and demographic patterns.

You can extend this analysis further using the data, schema, and field IDs for FAIR and reproducible research.